# Helmet Detection Dataset - Exploratory Data Analysis

This notebook performs EDA on the motorcycle helmet compliance detection dataset.

In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import yaml

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 1. Dataset Structure

In [ ]:
# Load data configuration
with open('configs/data.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"  Classes: {config['names']}")
print(f"  Number of classes: {config['nc']}")
print(f"  Train path: {config['path']}/{config['train']}")
print(f"  Val path: {config['path']}/{config['val']}")
print(f"  Test path: {config['path']}/{config['test']}")

## 2. Image Statistics

In [ ]:
def get_image_stats(directory):
    """Get statistics for images in a directory."""
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    images = []
    
    for ext in image_exts:
        images.extend(list(Path(directory).glob(f'**/*{ext}')))
    
    if not images:
        return {'count': 0, 'sizes': [], 'aspect_ratios': []}
    
    sizes = []
    aspect_ratios = []
    
    for img_path in images:
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                sizes.append((w, h))
                aspect_ratios.append(w / h)
        except Exception as e:
            print(f"Error reading {img_path}: {e}")
    
    return {
        'count': len(images),
        'sizes': sizes,
        'aspect_ratios': aspect_ratios,
        'widths': [s[0] for s in sizes],
        'heights': [s[1] for s in sizes]
    }

# Get stats for each split
base_path = Path(config['path'])
splits = ['train', 'val', 'test']

split_stats = {}
for split in splits:
    img_dir = base_path / split / 'images'
    if img_dir.exists():
        split_stats[split] = get_image_stats(img_dir)
        print(f"\n{split.upper()} split:")
        print(f"  Images: {split_stats[split]['count']}")
        if split_stats[split]['sizes']:
            print(f"  Avg width: {np.mean(split_stats[split]['widths']):.0f}px")
            print(f"  Avg height: {np.mean(split_stats[split]['heights']):.0f}px")

## 3. Class Distribution Analysis

In [ ]:
def count_classes(label_dir):
    """Count class occurrences in label files."""
    class_counts = Counter()
    
    for label_file in Path(label_dir).glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1
    
    return class_counts

# Count classes for each split
class_names = config['names']
class_distributions = {}

for split in splits:
    label_dir = base_path / split / 'labels'
    if label_dir.exists():
        class_counts = count_classes(label_dir)
        class_distributions[split] = class_counts
        
        print(f"\n{split.upper()} class distribution:")
        for class_id, count in sorted(class_counts.items()):
            class_name = class_names.get(class_id, f'class_{class_id}')
            print(f"  {class_name}: {count}")

## 4. Visualization

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Image count per split
split_counts = [split_stats[s]['count'] if s in split_stats else 0 for s in splits]
axes[0, 0].bar(splits, split_counts, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[0, 0].set_title('Image Count per Split', fontsize=12)
axes[0, 0].set_ylabel('Number of Images')
for i, v in enumerate(split_counts):
    axes[0, 0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Plot 2: Class distribution (stacked bar)
if class_distributions:
    all_classes = sorted(set(c for dist in class_distributions.values() for c in dist.keys()))
    x = np.arange(len(splits))
    width = 0.6
    
    bottom = np.zeros(len(splits))
    for class_id in all_classes:
        counts = [class_distributions.get(s, {}).get(class_id, 0) for s in splits]
        class_name = class_names.get(class_id, f'class_{class_id}')
        axes[0, 1].bar(x, counts, width, label=class_name, bottom=bottom)
        bottom += np.array(counts)
    
    axes[0, 1].set_title('Class Distribution per Split', fontsize=12)
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(splits)
    axes[0, 1].legend()

# Plot 3: Image width distribution
if 'train' in split_stats and split_stats['train']['widths']:
    axes[1, 0].hist(split_stats['train']['widths'], bins=20, color='#2196F3', alpha=0.7)
    axes[1, 0].set_title('Training Image Width Distribution', fontsize=12)
    axes[1, 0].set_xlabel('Width (pixels)')
    axes[1, 0].set_ylabel('Count')

# Plot 4: Image height distribution
if 'train' in split_stats and split_stats['train']['heights']:
    axes[1, 1].hist(split_stats['train']['heights'], bins=20, color='#4CAF50', alpha=0.7)
    axes[1, 1].set_title('Training Image Height Distribution', fontsize=12)
    axes[1, 1].set_xlabel('Height (pixels)')
    axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('notebooks/eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to notebooks/eda_analysis.png")

## 5. Sample Annotations Visualization

In [ ]:
def visualize_annotations(image_path, label_path, class_names, ax):
    """Visualize image with bounding box annotations."""
    # Load image
    img = Image.open(image_path)
    ax.imshow(img)
    
    # Get image dimensions
    img_w, img_h = img.size
    
    # Colors for each class
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    # Read annotations
    if Path(label_path).exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center = float(parts[1]) * img_w
                    y_center = float(parts[2]) * img_h
                    width = float(parts[3]) * img_w
                    height = float(parts[4]) * img_h
                    
                    # Convert to corner coordinates
                    x1 = x_center - width/2
                    y1 = y_center - height/2
                    x2 = x_center + width/2
                    y2 = y_center + height/2
                    
                    # Draw bounding box
                    color = colors[class_id % len(colors)]
                    rect = plt.Rectangle((x1, y1), width, height, 
                                       fill=False, edgecolor=color, linewidth=2)
                    ax.add_patch(rect)
                    
                    # Add label
                    class_name = class_names.get(class_id, f'class_{class_id}')
                    ax.text(x1, y1-5, class_name, color=color, fontsize=10,
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.axis('off')

# Visualize sample images from each split
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, split in enumerate(splits):
    img_dir = base_path / split / 'images'
    label_dir = base_path / split / 'labels'
    
    if img_dir.exists():
        # Get first image
        img_files = list(img_dir.glob('*'))[:1]
        if img_files:
            img_path = img_files[0]
            label_path = label_dir / (img_path.stem + '.txt')
            visualize_annotations(img_path, label_path, class_names, axes[idx])
            axes[idx].set_title(f'{split.upper()} Split Sample')

plt.tight_layout()
plt.savefig('notebooks/sample_annotations.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample annotations saved to notebooks/sample_annotations.png")

## 6. Summary Statistics

In [ ]:
# Generate summary report
print("="*60)
print("DATASET SUMMARY REPORT")
print("="*60)

print(f"\nDataset Path: {config['path']}")
print(f"Number of Classes: {config['nc']}")
print(f"Class Names: {config['names']}")

print("\nSplit Statistics:")
for split in splits:
    if split in split_stats:
        print(f"  {split.upper()}:")
        print(f"    Images: {split_stats[split]['count']}")
        if split_stats[split]['sizes']:
            print(f"    Avg Resolution: {np.mean(split_stats[split]['widths']):.0f}x{np.mean(split_stats[split]['heights']):.0f}")

print("\nClass Distribution:")
for split in splits:
    if split in class_distributions:
        total = sum(class_distributions[split].values())
        print(f"  {split.upper()} (total annotations: {total}):")
        for class_id, count in sorted(class_distributions[split].items()):
            class_name = class_names.get(class_id, f'class_{class_id}')
            pct = (count / total * 100) if total > 0 else 0
            print(f"    {class_name}: {count} ({pct:.1f}%)")

print("\n" + "="*60)